In [5]:
# %%
import warnings
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

BASE_PATH = '/Users/egorilin/Desktop/COMA'
BASE = Path(BASE_PATH)

RANDOM_STATE = 42
TEST_SIZE = 0.2

CB_KW = dict(iterations=500, learning_rate=0.05, depth=6,
             verbose=False, random_seed=RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

TARGET_COLUMNS = [
    'S. aureus ATCC 43300 Activity MIC, мг/л',
    'S. aureus ATCC 43300 Activity MBC, мг/л',
    'E. coli ATCC 25922 Activity MIC, мг/л',
    'E. coli ATCC 25922 Activity MBC, мг/л',
]

# 'biocides' — 12 фич (без диполей); 'conc' — 15 фич (с диполями)
DATASET_MODE = 'conc'

FEATURE_COLS_FULL = [
    'Molecular weight', 'LogP', 'mmff94', 'gasteiger', 'eem2015bm',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS_BIO = [
    'Molecular weight', 'LogP',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS = FEATURE_COLS_BIO if DATASET_MODE == 'biocides' else FEATURE_COLS_FULL
DIPOLE_COLS = ['mmff94', 'gasteiger', 'eem2015bm']

DESC_OUT_DIR = BASE / 'descriptors_test_predictions'
EMB_OUT_DIR  = BASE / 'embeddings_test_predictions'
DESC_OUT_DIR.mkdir(exist_ok=True)
EMB_OUT_DIR.mkdir(exist_ok=True)

print(f'Dataset mode: {DATASET_MODE}  |  {len(FEATURE_COLS)} фич')
print(f'Output dirs:\n  {DESC_OUT_DIR}\n  {EMB_OUT_DIR}')

Device: cpu
Dataset mode: conc  |  15 фич
Output dirs:
  /Users/egorilin/Desktop/COMA/descriptors_test_predictions
  /Users/egorilin/Desktop/COMA/embeddings_test_predictions


In [12]:
target

,Unnamed: 0,General formula,Molecular weight,Gram_aureus,Gram_coli,LogP,"S. aureus ATCC 43300 Activity MIC, мг/л","S. aureus ATCC 43300 Activity MBC, мг/л","E. coli ATCC 25922 Activity MIC, мг/л","E. coli ATCC 25922 Activity MBC, мг/л"
0,0,CCCCCCCCCC[N+]1=CC=CC(OC2=CC=C(OC3=C[N+](CCCCC...,706.7,0,1,2.634,1.0,4.0,2.0,4.0
1,1,CCCCCCCCCCC[N+]1=CC=CC(OC2=CC=C(OC3=C[N+](CCCC...,734.7,0,1,3.523,4.0,4.0,500.0,500.0
2,2,CCCCCCCCCCCC[N+]1=CC=CC(OC2=CC=C(OC3=C[N+](CCC...,762.8,0,1,3.810,32.0,125.0,8.0,8.0
3,3,CCCCCCCCCCC[N+](C=C1)=CC=C1OC2=CC=C(OC3=CC=[N+...,734.7,0,1,3.523,16.0,32.0,16.0,16.0
4,4,CCCCCCCCCC[N+](C=CC=C1)=C1OC2=CC=C(OC3=[N+](CC...,706.7,0,1,2.634,2.0,8.0,4.0,4.0
...,...,...,...,...,...,...,...,...,...,...
103,103,CC(C1=CC=C(OC2=[N+](CCCCCC)C=CC=C2)C=C1)(C)C3=...,806.6,0,1,1.935,0.5,1.0,8.0,16.0
104,104,CC(C1=CC=C(OC2=[N+](CCCCCCC)C=CC=C2)C=C1)(C)C3...,834.7,0,1,2.824,2.0,2.0,4.0,4.0
105,105,CC(C1=CC=C(OC2=[N+](CCCCCCCC)C=CC=C2)C=C1)(C)C...,862.7,0,1,3.714,4.0,4.0,4.0,4.0
106,106,CC(C1=CC=C(OC2=[N+](CCCCCCCCCC)C=CC=C2)C=C1)(C...,918.8,0,1,5.492,0.5,0.5,32.0,125.0


In [14]:
# %%
target = pd.read_excel(f'{BASE_PATH}/Data_biocides.xlsx')
desc_df = pd.read_excel(f'{BASE_PATH}/Data_prepare_biocides_molformer_new_embeddings.xlsx')
data_drop = pd.read_excel(f'{BASE_PATH}/Test_molformer_original_biocides.xlsx').drop(
    ['Unnamed: 0'], axis=1
)
dipoles_df = pd.read_csv(f'{BASE_PATH}/Train_dipole_moments.csv')

print(f'target:     {target.shape}')
print(f'desc_df:    {desc_df.shape}')
print(f'data_drop:  {data_drop.shape}')
print(f'dipoles_df: {dipoles_df.shape}  | cols: {list(dipoles_df.columns)}')

# --- Базовые фичи (LogP, Molecular weight) берём из target по позиции ---
BASE_FEATURE_COLS = ['Molecular weight', 'LogP']
missing_in_target = [c for c in BASE_FEATURE_COLS if c not in target.columns]
if missing_in_target:
    raise ValueError(f'В target нет колонок: {missing_in_target}. '
                     f'Колонки target: {list(target.columns)[:20]}')

desc_df = desc_df.reset_index(drop=True)
target_reset = target.reset_index(drop=True)

# выкидываем эти колонки из desc_df, если уже есть, и подставляем из target по позиции
desc_df = desc_df.drop(columns=[c for c in BASE_FEATURE_COLS if c in desc_df.columns])
n_base = min(len(desc_df), len(target_reset))
if len(desc_df) != len(target_reset):
    print(f'  ⚠️  Длины desc_df({len(desc_df)}) и target({len(target_reset)}) не совпадают, '
          f'возьму первые {n_base}')

for c in BASE_FEATURE_COLS:
    desc_df[c] = np.nan
    desc_df.loc[:n_base - 1, c] = target_reset[c].iloc[:n_base].values

print('\nПодставлено из target:')
for c in BASE_FEATURE_COLS:
    n_nan = desc_df[c].isna().sum()
    print(f'  {c}: {len(desc_df) - n_nan}/{len(desc_df)} заполнено  ({n_nan} NaN)')

# --- Подстановка диполей по позиции из csv ---
first_col = dipoles_df.columns[0]
if dipoles_df[first_col].dtype.kind in 'iu' or first_col.lower() in ('unnamed: 0', 'index', 'id', '№'):
    print(f'\n  Отбрасываю служебную первую колонку csv: "{first_col}"')
    dipoles_df = dipoles_df.drop(columns=[first_col])

have_dipoles = [c for c in DIPOLE_COLS if c in dipoles_df.columns]
miss_dipoles = [c for c in DIPOLE_COLS if c not in dipoles_df.columns]
print(f'  Диполи в csv: {have_dipoles}')
if miss_dipoles:
    print(f'  ⚠️  Нет в csv диполей: {miss_dipoles}')

n_dip = min(len(desc_df), len(dipoles_df))
if len(dipoles_df) != len(desc_df):
    print(f'  ⚠️  Длины desc_df({len(desc_df)}) и dipoles_df({len(dipoles_df)}) не совпадают, '
          f'возьму первые {n_dip}')

desc_df = desc_df.drop(columns=[c for c in DIPOLE_COLS if c in desc_df.columns])
dipoles_aligned = dipoles_df[have_dipoles].reset_index(drop=True)

for c in have_dipoles:
    desc_df[c] = np.nan
    desc_df.loc[:n_dip - 1, c] = dipoles_aligned[c].iloc[:n_dip].values

print('\nПокрытие диполей после подстановки:')
for c in have_dipoles:
    n_nan = desc_df[c].isna().sum()
    print(f'  {c}: {len(desc_df) - n_nan}/{len(desc_df)} заполнено  ({n_nan} NaN)')

# --- Таргеты тоже должны быть в desc_df для Ячейки 4. Если их там нет — берём из target по позиции ---
for c in TARGET_COLUMNS:
    if c not in desc_df.columns:
        if c not in target_reset.columns:
            raise ValueError(f'Нет таргета "{c}" ни в desc_df, ни в target')
        desc_df[c] = np.nan
        n_tg = min(len(desc_df), len(target_reset))
        desc_df.loc[:n_tg - 1, c] = target_reset[c].iloc[:n_tg].values

# --- Финальные проверки ---
missing_desc = [c for c in FEATURE_COLS if c not in desc_df.columns]
if missing_desc:
    raise ValueError(f'Нет колонок в desc_df после всех подстановок: {missing_desc}')

for c in TARGET_COLUMNS:
    if c not in target.columns:
        raise ValueError(f'Нет таргета в target: {c}')

print('\n✓ Все нужные колонки на месте')
print(f'  FEATURE_COLS ({len(FEATURE_COLS)}): {FEATURE_COLS}')

target:     (108, 10)
desc_df:    (108, 769)
data_drop:  (108, 768)
dipoles_df: (108, 4)  | cols: ['Unnamed: 0', 'mmff94', 'gasteiger', 'eem2015bm']

Подставлено из target:
  Molecular weight: 108/108 заполнено  (0 NaN)
  LogP: 101/108 заполнено  (7 NaN)

  Отбрасываю служебную первую колонку csv: "Unnamed: 0"
  Диполи в csv: ['mmff94', 'gasteiger', 'eem2015bm']

Покрытие диполей после подстановки:
  mmff94: 108/108 заполнено  (0 NaN)
  gasteiger: 108/108 заполнено  (0 NaN)
  eem2015bm: 108/108 заполнено  (0 NaN)

✓ Все нужные колонки на месте
  FEATURE_COLS (15): ['Molecular weight', 'LogP', 'mmff94', 'gasteiger', 'eem2015bm', 'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85', 'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729', 'Molformer_337', 'Molformer_686']


In [16]:
# %%
class AdvancedContrastiveEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 384, 128],
                 projection_dim=128, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim
        self.encoder = nn.Sequential(*layers)
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], projection_dim),
            nn.BatchNorm1d(projection_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(projection_dim, projection_dim),
            nn.BatchNorm1d(projection_dim),
        )

    def forward(self, x, return_projection=True):
        features = self.encoder(x)
        if return_projection:
            return features, self.projection_head(features)
        return features


encoder = AdvancedContrastiveEncoder(
    input_dim=768, hidden_dims=[512, 384, 256],
    projection_dim=128, dropout=0.3,
).to(device)
encoder.load_state_dict(
    torch.load(f'{BASE_PATH}/encoder_borderline_smote.pth', map_location=device)
)
encoder.eval()

with open(f'{BASE_PATH}/scaler_classification.pkl', 'rb') as f:
    scaler = pickle.load(f)

data_scaled = scaler.transform(data_drop)
X_tensor = torch.FloatTensor(data_scaled).to(device)
with torch.no_grad():
    molecular_embeddings = encoder(X_tensor, return_projection=False).cpu().numpy()

print(f'✓ molecular_embeddings: {molecular_embeddings.shape}')

✓ molecular_embeddings: (108, 256)


In [17]:
# %%
desc_all_records  = []
desc_metrics_rows = []

for target_col in TARGET_COLUMNS:
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    tag = f"{organism.replace(' ', '_').replace('.', '')}_{short_name}"

    cols_needed = FEATURE_COLS + [target_col]
    valid_mask = desc_df[cols_needed].notna().all(axis=1)
    sub = desc_df.loc[valid_mask, cols_needed].reset_index().rename(
        columns={'index': 'mol_idx'}
    )

    if len(sub) < 10:
        print(f'\n⚠️  {target_col}: слишком мало валидных образцов ({len(sub)}), пропуск')
        continue

    X      = sub[FEATURE_COLS].values
    y_raw  = sub[target_col].values
    y_log2 = np.log2(y_raw)
    mol_idx = sub['mol_idx'].values

    idx_arr = np.arange(len(sub))
    train_idx, test_idx = train_test_split(
        idx_arr, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_log2[train_idx], y_log2[test_idx]
    mol_test = mol_idx[test_idx]

    model = CatBoostRegressor(**CB_KW)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    pr, _ = pearsonr(y_test, y_pred)

    print(f'\n{target_col}')
    print(f'  N_train={len(y_train)}  N_test={len(y_test)}')
    print(f'  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  Pearson={pr:.4f}')

    per_target_df = pd.DataFrame({
        'mol_idx':       mol_test,
        'organism':      organism,
        'metric_type':   short_name,
        'target_column': target_col,
        'y_true_log2':   y_test,
        'y_pred_log2':   y_pred,
        'y_true_raw':    2.0 ** y_test,
        'y_pred_raw':    2.0 ** y_pred,
    })
    out_path = DESC_OUT_DIR / f'test_pred__{tag}.csv'
    per_target_df.to_csv(out_path, index=False)
    print(f'  ✓ {out_path.name}')

    desc_all_records.append(per_target_df)
    desc_metrics_rows.append({
        'Approach': 'descriptors',
        'Organism': organism, 'Target': short_name,
        'N_train': len(y_train), 'N_test': len(y_test),
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'Pearson_r': pr,
    })

desc_all_df = pd.concat(desc_all_records, ignore_index=True)
desc_all_df.to_csv(DESC_OUT_DIR / 'test_pred__ALL_targets_concat.csv', index=False)

desc_metrics_df = pd.DataFrame(desc_metrics_rows)
desc_metrics_df.to_csv(DESC_OUT_DIR / 'metrics_per_target.csv', index=False)

y_true_all = desc_all_df['y_true_log2'].values
y_pred_all = desc_all_df['y_pred_log2'].values
desc_overall = {
    'Approach': 'descriptors',
    'N_test_total': len(y_true_all),
    'MAE':  mean_absolute_error(y_true_all, y_pred_all),
    'RMSE': np.sqrt(mean_squared_error(y_true_all, y_pred_all)),
    'R2':   r2_score(y_true_all, y_pred_all),
    'Pearson_r': pearsonr(y_true_all, y_pred_all)[0],
}
pd.DataFrame([desc_overall]).to_csv(DESC_OUT_DIR / 'metrics_overall.csv', index=False)

print('\n=== DESCRIPTORS — overall (concat) ===')
for k, v in desc_overall.items():
    print(f'  {k}: {v}')
print(f'\n✓ {DESC_OUT_DIR}')


S. aureus ATCC 43300 Activity MIC, мг/л
  N_train=80  N_test=21
  MAE=1.4996  RMSE=1.9907  R²=-0.2376  Pearson=0.2226
  ✓ test_pred__S_aureus_ATCC_43300_MIC.csv

S. aureus ATCC 43300 Activity MBC, мг/л
  N_train=80  N_test=21
  MAE=1.4771  RMSE=1.9861  R²=-0.5540  Pearson=0.0473
  ✓ test_pred__S_aureus_ATCC_43300_MBC.csv

E. coli ATCC 25922 Activity MIC, мг/л
  N_train=80  N_test=21
  MAE=1.6652  RMSE=2.1581  R²=-0.0433  Pearson=0.2484
  ✓ test_pred__E_coli_ATCC_25922_MIC.csv

E. coli ATCC 25922 Activity MBC, мг/л
  N_train=80  N_test=21
  MAE=1.6357  RMSE=2.2201  R²=-0.0542  Pearson=0.2424
  ✓ test_pred__E_coli_ATCC_25922_MBC.csv

=== DESCRIPTORS — overall (concat) ===
  Approach: descriptors
  N_test_total: 84
  MAE: 1.56941149809936
  RMSE: 2.0912383136622528
  R2: 0.08005350526558741
  Pearson_r: 0.44275172381306044

✓ /Users/egorilin/Desktop/COMA/descriptors_test_predictions


In [18]:
# %%
emb_all_records  = []
emb_metrics_rows = []

for target_col in TARGET_COLUMNS:
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    tag = f"{organism.replace(' ', '_').replace('.', '')}_{short_name}"

    valid_mask = ~target[target_col].isna()
    valid_idx_arr = np.where(valid_mask.values)[0]

    if len(valid_idx_arr) < 10:
        print(f'\n⚠️  {target_col}: слишком мало валидных образцов ({len(valid_idx_arr)}), пропуск')
        continue

    X      = molecular_embeddings[valid_idx_arr]
    y_raw  = target[target_col].values[valid_idx_arr]
    y_log2 = np.log2(y_raw)

    pos_arr = np.arange(len(valid_idx_arr))
    train_pos, test_pos = train_test_split(
        pos_arr, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    X_train, X_test = X[train_pos], X[test_pos]
    y_train, y_test = y_log2[train_pos], y_log2[test_pos]
    mol_test = valid_idx_arr[test_pos]

    model = CatBoostRegressor(**CB_KW)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    pr, _ = pearsonr(y_test, y_pred)

    print(f'\n{target_col}')
    print(f'  N_train={len(y_train)}  N_test={len(y_test)}')
    print(f'  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  Pearson={pr:.4f}')

    per_target_df = pd.DataFrame({
        'mol_idx':       mol_test,
        'organism':      organism,
        'metric_type':   short_name,
        'target_column': target_col,
        'y_true_log2':   y_test,
        'y_pred_log2':   y_pred,
        'y_true_raw':    2.0 ** y_test,
        'y_pred_raw':    2.0 ** y_pred,
    })
    out_path = EMB_OUT_DIR / f'test_pred__{tag}.csv'
    per_target_df.to_csv(out_path, index=False)
    print(f'  ✓ {out_path.name}')

    emb_all_records.append(per_target_df)
    emb_metrics_rows.append({
        'Approach': 'embeddings',
        'Organism': organism, 'Target': short_name,
        'N_train': len(y_train), 'N_test': len(y_test),
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'Pearson_r': pr,
    })

emb_all_df = pd.concat(emb_all_records, ignore_index=True)
emb_all_df.to_csv(EMB_OUT_DIR / 'test_pred__ALL_targets_concat.csv', index=False)

emb_metrics_df = pd.DataFrame(emb_metrics_rows)
emb_metrics_df.to_csv(EMB_OUT_DIR / 'metrics_per_target.csv', index=False)

y_true_all = emb_all_df['y_true_log2'].values
y_pred_all = emb_all_df['y_pred_log2'].values
emb_overall = {
    'Approach': 'embeddings',
    'N_test_total': len(y_true_all),
    'MAE':  mean_absolute_error(y_true_all, y_pred_all),
    'RMSE': np.sqrt(mean_squared_error(y_true_all, y_pred_all)),
    'R2':   r2_score(y_true_all, y_pred_all),
    'Pearson_r': pearsonr(y_true_all, y_pred_all)[0],
}
pd.DataFrame([emb_overall]).to_csv(EMB_OUT_DIR / 'metrics_overall.csv', index=False)

print('\n=== EMBEDDINGS — overall (concat) ===')
for k, v in emb_overall.items():
    print(f'  {k}: {v}')
print(f'\n✓ {EMB_OUT_DIR}')


S. aureus ATCC 43300 Activity MIC, мг/л
  N_train=86  N_test=22
  MAE=1.0034  RMSE=1.3343  R²=0.0297  Pearson=0.4562
  ✓ test_pred__S_aureus_ATCC_43300_MIC.csv

S. aureus ATCC 43300 Activity MBC, мг/л
  N_train=86  N_test=22
  MAE=1.3542  RMSE=1.8623  R²=0.0107  Pearson=0.4451
  ✓ test_pred__S_aureus_ATCC_43300_MBC.csv

E. coli ATCC 25922 Activity MIC, мг/л
  N_train=86  N_test=22
  MAE=1.0102  RMSE=1.3993  R²=0.1665  Pearson=0.6118
  ✓ test_pred__E_coli_ATCC_25922_MIC.csv

E. coli ATCC 25922 Activity MBC, мг/л
  N_train=86  N_test=22
  MAE=0.9995  RMSE=1.2601  R²=0.3407  Pearson=0.6727
  ✓ test_pred__E_coli_ATCC_25922_MBC.csv

=== EMBEDDINGS — overall (concat) ===
  Approach: embeddings
  N_test_total: 88
  MAE: 1.0918172662515777
  RMSE: 1.482751237610632
  R2: 0.3836371452224735
  Pearson_r: 0.6996582706049863

✓ /Users/egorilin/Desktop/COMA/embeddings_test_predictions


In [19]:
# %%
summary = pd.concat([desc_metrics_df, emb_metrics_df], ignore_index=True)
print('Per-target:')
print(summary.to_string(index=False))

overall = pd.DataFrame([desc_overall, emb_overall])
print('\nOverall (concat across all 4 targets):')
print(overall.to_string(index=False))

summary.to_csv(BASE / 'comparison_per_target.csv', index=False)
overall.to_csv(BASE / 'comparison_overall.csv', index=False)
print(f'\n✓ {BASE}/comparison_per_target.csv')
print(f'✓ {BASE}/comparison_overall.csv')

Per-target:
   Approach             Organism Target  N_train  N_test      MAE     RMSE        R2  Pearson_r
descriptors S. aureus ATCC 43300    MIC       80      21 1.499614 1.990659 -0.237649   0.222565
descriptors S. aureus ATCC 43300    MBC       80      21 1.477137 1.986071 -0.554022   0.047284
descriptors   E. coli ATCC 25922    MIC       80      21 1.665188 2.158051 -0.043290   0.248387
descriptors   E. coli ATCC 25922    MBC       80      21 1.635707 2.220074 -0.054160   0.242405
 embeddings S. aureus ATCC 43300    MIC       86      22 1.003393 1.334261  0.029682   0.456203
 embeddings S. aureus ATCC 43300    MBC       86      22 1.354210 1.862322  0.010676   0.445089
 embeddings   E. coli ATCC 25922    MIC       86      22 1.010180 1.399272  0.166473   0.611821
 embeddings   E. coli ATCC 25922    MBC       86      22 0.999486 1.260058  0.340718   0.672748

Overall (concat across all 4 targets):
   Approach  N_test_total      MAE     RMSE       R2  Pearson_r
descriptors         

# Multitask on the new эмбеддингм

In [20]:
# %%
from openai import OpenAI

OPENROUTER_API_KEY = "[REDACTED_API_KEY]"
EMBEDDING_MODEL = "openai/text-embedding-3-small"

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

def get_embedding(text, model=EMBEDDING_MODEL):
    text = text.replace("\n", " ")
    resp = client.embeddings.create(input=[text], model=model)
    return resp.data[0].embedding

CLASS_DESCRIPTIONS = {
    'S. aureus ATCC 43300_MIC': (
        "Minimum Inhibitory Concentration against Staphylococcus aureus ATCC 43300, "
        "a methicillin-resistant Gram-positive bacterium. MIC measures the lowest "
        "concentration that inhibits visible bacterial growth."
    ),
    'S. aureus ATCC 43300_MBC': (
        "Minimum Bactericidal Concentration against Staphylococcus aureus ATCC 43300, "
        "a methicillin-resistant Gram-positive bacterium. MBC measures the lowest "
        "concentration that kills 99.9% of bacteria."
    ),
    'E. coli ATCC 25922_MIC': (
        "Minimum Inhibitory Concentration against Escherichia coli ATCC 25922, "
        "a Gram-negative reference strain. MIC measures the lowest concentration "
        "that inhibits visible bacterial growth."
    ),
    'E. coli ATCC 25922_MBC': (
        "Minimum Bactericidal Concentration against Escherichia coli ATCC 25922, "
        "a Gram-negative reference strain. MBC measures the lowest concentration "
        "that kills 99.9% of bacteria."
    ),
}

# проверка соединения + получение эмбеддингов
print('Получение OpenAI эмбеддингов меток через OpenRouter...')
class_embeddings = {}
for key, desc in CLASS_DESCRIPTIONS.items():
    emb = get_embedding(desc)
    class_embeddings[key] = np.array(emb, dtype=np.float32)
    print(f'  {key}: dim={len(emb)}')

openai_emb_dim = len(next(iter(class_embeddings.values())))
print(f'\n✓ OpenAI label embeddings: {openai_emb_dim}-dim')

Получение OpenAI эмбеддингов меток через OpenRouter...
  S. aureus ATCC 43300_MIC: dim=1536
  S. aureus ATCC 43300_MBC: dim=1536
  E. coli ATCC 25922_MIC: dim=1536
  E. coli ATCC 25922_MBC: dim=1536

✓ OpenAI label embeddings: 1536-dim


In [25]:
# %%
from catboost import Pool

# Папки вывода
DESC_CAT_OUT_DIR = BASE / 'descriptors_cat_test_predictions'
DESC_OAI_OUT_DIR = BASE / 'descriptors_openai_test_predictions'
EMB_CAT_OUT_DIR  = BASE / 'embeddings_cat_test_predictions'
EMB_OAI_OUT_DIR  = BASE / 'embeddings_openai_test_predictions'
for d in [DESC_CAT_OUT_DIR, DESC_OAI_OUT_DIR, EMB_CAT_OUT_DIR, EMB_OAI_OUT_DIR]:
    d.mkdir(exist_ok=True)

# --- Multi-task на ЭМБЕДДИНГАХ ---
emb_dim = molecular_embeddings.shape[1]
print(f'molecular_embeddings dim = {emb_dim}')

emb_records = []
for target_col in TARGET_COLUMNS:
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    key = f'{organism}_{short_name}'
    valid_idx = np.where(~target[target_col].isna().values)[0]
    for mi in valid_idx:
        emb_records.append({
            'mol_idx': int(mi),
            'organism': organism,
            'metric_type': short_name,
            'target_column': target_col,
            'class_key': key,
            'y_log2': float(np.log2(target[target_col].values[mi])),
        })
emb_pairs_df = pd.DataFrame(emb_records).reset_index(drop=True)
print(f'Emb multi-task: {len(emb_pairs_df)} пар')

# --- Multi-task на ДЕСКРИПТОРАХ ---
# берём только валидные строки (где есть фичи и есть таргет)
desc_records_rows = []
for target_col in TARGET_COLUMNS:
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    key = f'{organism}_{short_name}'

    cols_needed = FEATURE_COLS + [target_col]
    valid_mask = desc_df[cols_needed].notna().all(axis=1)
    sub = desc_df.loc[valid_mask, cols_needed].copy()
    sub['mol_idx']       = sub.index.values
    sub['organism']      = organism
    sub['metric_type']   = short_name
    sub['target_column'] = target_col
    sub['class_key']     = key
    sub['y_log2']        = np.log2(sub[target_col].values)
    desc_records_rows.append(sub)

desc_pairs_df = pd.concat(desc_records_rows, ignore_index=True)
print(f'Desc multi-task: {len(desc_pairs_df)} пар')


def make_split(n_pairs):
    return train_test_split(np.arange(n_pairs), test_size=TEST_SIZE, random_state=RANDOM_STATE)


def per_target_metrics(test_df, approach_name):
    rows = []
    for tc in TARGET_COLUMNS:
        sub = test_df[test_df['target_column'] == tc]
        if len(sub) < 2:
            continue
        short_name = tc.split(' Activity ')[1].replace(', мг/л', '')
        organism   = tc.split(' Activity ')[0]
        yt, yp = sub['y_true_log2'].values, sub['y_pred_log2'].values
        rows.append({
            'Approach': approach_name,
            'Organism': organism, 'Target': short_name,
            'N_test': len(sub),
            'MAE':  mean_absolute_error(yt, yp),
            'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'R2':   r2_score(yt, yp),
            'Pearson_r': pearsonr(yt, yp)[0],
        })
    return pd.DataFrame(rows)


def save_multitask_outputs(test_df, approach_name, out_dir):
    cols = ['mol_idx','organism','metric_type','target_column',
            'y_true_log2','y_pred_log2','y_true_raw','y_pred_raw']
    for tc in TARGET_COLUMNS:
        sub = test_df[test_df['target_column'] == tc]
        if len(sub) < 2:
            continue
        short_name = tc.split(' Activity ')[1].replace(', мг/л', '')
        organism   = tc.split(' Activity ')[0]
        tag = f"{organism.replace(' ', '_').replace('.', '')}_{short_name}"
        sub[cols].to_csv(out_dir / f'test_pred__{tag}.csv', index=False)

    test_df[cols].to_csv(out_dir / 'test_pred__ALL_targets_concat.csv', index=False)
    pt = per_target_metrics(test_df, approach_name)
    pt.to_csv(out_dir / 'metrics_per_target.csv', index=False)

    yt, yp = test_df['y_true_log2'].values, test_df['y_pred_log2'].values
    overall = {
        'Approach': approach_name,
        'N_test_total': len(test_df),
        'MAE':  mean_absolute_error(yt, yp),
        'RMSE': np.sqrt(mean_squared_error(yt, yp)),
        'R2':   r2_score(yt, yp),
        'Pearson_r': pearsonr(yt, yp)[0],
    }
    pd.DataFrame([overall]).to_csv(out_dir / 'metrics_overall.csv', index=False)
    return pt, overall

molecular_embeddings dim = 256
Emb multi-task: 432 пар
Desc multi-task: 404 пар


In [26]:
# %%
# Сплит для дескрипторного multi-task
train_idx_d, test_idx_d = make_split(len(desc_pairs_df))
y_full_d = desc_pairs_df['y_log2'].values

# === 3) desc + categorical ===
print('\n--- desc + categorical ---')
X_d_cat = desc_pairs_df[FEATURE_COLS].copy()
X_d_cat['organism']    = desc_pairs_df['organism'].values
X_d_cat['metric_type'] = desc_pairs_df['metric_type'].values

X_tr = X_d_cat.iloc[train_idx_d].reset_index(drop=True)
X_te = X_d_cat.iloc[test_idx_d].reset_index(drop=True)
y_tr = y_full_d[train_idx_d]
y_te = y_full_d[test_idx_d]

train_pool = Pool(X_tr, label=y_tr, cat_features=['organism', 'metric_type'])
test_pool  = Pool(X_te, label=y_te, cat_features=['organism', 'metric_type'])
m = CatBoostRegressor(**CB_KW); m.fit(train_pool)
y_pred = m.predict(test_pool)

meta = desc_pairs_df.iloc[test_idx_d].reset_index(drop=True).copy()
meta['y_true_log2'] = y_te
meta['y_pred_log2'] = y_pred
meta['y_true_raw']  = 2.0 ** y_te
meta['y_pred_raw']  = 2.0 ** y_pred
desc_cat_metrics_df, desc_cat_overall = save_multitask_outputs(meta, 'desc+categorical', DESC_CAT_OUT_DIR)
print(f"Overall  MAE={desc_cat_overall['MAE']:.4f}  R²={desc_cat_overall['R2']:.4f}  "
      f"Pearson={desc_cat_overall['Pearson_r']:.4f}")

# === 4) desc + OpenAI ===
print('\n--- desc + openai ---')
label_emb_d = np.stack(
    [class_embeddings[k] for k in desc_pairs_df['class_key'].values]
).astype(np.float32)
X_d_oai = np.hstack([desc_pairs_df[FEATURE_COLS].values, label_emb_d])

X_tr = X_d_oai[train_idx_d]; X_te = X_d_oai[test_idx_d]
y_tr = y_full_d[train_idx_d]; y_te = y_full_d[test_idx_d]

m = CatBoostRegressor(**CB_KW); m.fit(X_tr, y_tr)
y_pred = m.predict(X_te)

meta = desc_pairs_df.iloc[test_idx_d].reset_index(drop=True).copy()
meta['y_true_log2'] = y_te
meta['y_pred_log2'] = y_pred
meta['y_true_raw']  = 2.0 ** y_te
meta['y_pred_raw']  = 2.0 ** y_pred
desc_oai_metrics_df, desc_oai_overall = save_multitask_outputs(meta, 'desc+openai', DESC_OAI_OUT_DIR)
print(f"Overall  MAE={desc_oai_overall['MAE']:.4f}  R²={desc_oai_overall['R2']:.4f}  "
      f"Pearson={desc_oai_overall['Pearson_r']:.4f}")


--- desc + categorical ---
Overall  MAE=1.0002  R²=0.6407  Pearson=0.8144

--- desc + openai ---
Overall  MAE=0.9913  R²=0.6180  Pearson=0.8014


In [27]:
# %%
train_idx_e, test_idx_e = make_split(len(emb_pairs_df))
y_full_e = emb_pairs_df['y_log2'].values
mol_emb_per_pair = molecular_embeddings[emb_pairs_df['mol_idx'].values]
emb_cols = [f'emb_{i}' for i in range(emb_dim)]

# === 5) emb + categorical ===
print('\n--- emb + categorical ---')
X_e_cat = pd.DataFrame(mol_emb_per_pair, columns=emb_cols)
X_e_cat['organism']    = emb_pairs_df['organism'].values
X_e_cat['metric_type'] = emb_pairs_df['metric_type'].values

X_tr = X_e_cat.iloc[train_idx_e].reset_index(drop=True)
X_te = X_e_cat.iloc[test_idx_e].reset_index(drop=True)
y_tr = y_full_e[train_idx_e]; y_te = y_full_e[test_idx_e]

train_pool = Pool(X_tr, label=y_tr, cat_features=['organism', 'metric_type'])
test_pool  = Pool(X_te, label=y_te, cat_features=['organism', 'metric_type'])
m = CatBoostRegressor(**CB_KW); m.fit(train_pool)
y_pred = m.predict(test_pool)

meta = emb_pairs_df.iloc[test_idx_e].reset_index(drop=True).copy()
meta['y_true_log2'] = y_te
meta['y_pred_log2'] = y_pred
meta['y_true_raw']  = 2.0 ** y_te
meta['y_pred_raw']  = 2.0 ** y_pred
emb_cat_metrics_df, emb_cat_overall = save_multitask_outputs(meta, 'emb+categorical', EMB_CAT_OUT_DIR)
print(f"Overall  MAE={emb_cat_overall['MAE']:.4f}  R²={emb_cat_overall['R2']:.4f}  "
      f"Pearson={emb_cat_overall['Pearson_r']:.4f}")

# === 6) emb + OpenAI ===
print('\n--- emb + openai ---')
label_emb_e = np.stack(
    [class_embeddings[k] for k in emb_pairs_df['class_key'].values]
).astype(np.float32)
X_e_oai = np.hstack([mol_emb_per_pair, label_emb_e])

X_tr = X_e_oai[train_idx_e]; X_te = X_e_oai[test_idx_e]
y_tr = y_full_e[train_idx_e]; y_te = y_full_e[test_idx_e]

m = CatBoostRegressor(**CB_KW); m.fit(X_tr, y_tr)
y_pred = m.predict(X_te)

meta = emb_pairs_df.iloc[test_idx_e].reset_index(drop=True).copy()
meta['y_true_log2'] = y_te
meta['y_pred_log2'] = y_pred
meta['y_true_raw']  = 2.0 ** y_te
meta['y_pred_raw']  = 2.0 ** y_pred
emb_oai_metrics_df, emb_oai_overall = save_multitask_outputs(meta, 'emb+openai', EMB_OAI_OUT_DIR)
print(f"Overall  MAE={emb_oai_overall['MAE']:.4f}  R²={emb_oai_overall['R2']:.4f}  "
      f"Pearson={emb_oai_overall['Pearson_r']:.4f}")


--- emb + categorical ---
Overall  MAE=1.0065  R²=0.6362  Pearson=0.8082

--- emb + openai ---
Overall  MAE=0.9873  R²=0.6824  Pearson=0.8430


In [23]:
# %%
from catboost import Pool

MULTI_CAT_OUT_DIR = BASE / 'embeddings_cat_test_predictions'
MULTI_CAT_OUT_DIR.mkdir(exist_ok=True)

emb_dim = molecular_embeddings.shape[1]
print(f'molecular_embeddings dim = {emb_dim}')

# --- Собираем multi-task датасет ---
records = []
for target_col in TARGET_COLUMNS:
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    key = f'{organism}_{short_name}'

    valid_idx = np.where(~target[target_col].isna().values)[0]
    for mi in valid_idx:
        records.append({
            'mol_idx':       int(mi),
            'organism':      organism,
            'metric_type':   short_name,
            'target_column': target_col,
            'class_key':     key,
            'y_log2':        float(np.log2(target[target_col].values[mi])),
        })

pairs_df = pd.DataFrame(records).reset_index(drop=True)
print(f'Multi-task датасет: {len(pairs_df)} пар (mol, target)')

# --- Фичи: molecular_embeddings[mol_idx] + organism + metric_type ---
mol_emb_per_pair = molecular_embeddings[pairs_df['mol_idx'].values]
emb_cols = [f'emb_{i}' for i in range(emb_dim)]
X_full = pd.DataFrame(mol_emb_per_pair, columns=emb_cols)
X_full['organism']    = pairs_df['organism'].values
X_full['metric_type'] = pairs_df['metric_type'].values
y_full = pairs_df['y_log2'].values

# --- Split (как в прошлых экспериментах) ---
idx_arr = np.arange(len(pairs_df))
train_idx, test_idx = train_test_split(
    idx_arr, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

X_train = X_full.iloc[train_idx].reset_index(drop=True)
X_test  = X_full.iloc[test_idx].reset_index(drop=True)
y_train = y_full[train_idx]
y_test  = y_full[test_idx]
meta_test = pairs_df.iloc[test_idx].reset_index(drop=True)

print(f'N_train={len(train_idx)}  N_test={len(test_idx)}')

train_pool = Pool(data=X_train, label=y_train, cat_features=['organism', 'metric_type'])
test_pool  = Pool(data=X_test,  label=y_test,  cat_features=['organism', 'metric_type'])

model_cat = CatBoostRegressor(**CB_KW)
model_cat.fit(train_pool)
y_pred_cat = model_cat.predict(test_pool)

mae  = mean_absolute_error(y_test, y_pred_cat)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat))
r2   = r2_score(y_test, y_pred_cat)
pr, _ = pearsonr(y_test, y_pred_cat)
print(f'\nOverall: MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  Pearson={pr:.4f}')

# --- Сохранение per-target и concat ---
test_df = meta_test.copy()
test_df['y_true_log2'] = y_test
test_df['y_pred_log2'] = y_pred_cat
test_df['y_true_raw']  = 2.0 ** y_test
test_df['y_pred_raw']  = 2.0 ** y_pred_cat

cat_metrics_rows = []
for target_col in TARGET_COLUMNS:
    sub = test_df[test_df['target_column'] == target_col]
    if len(sub) < 2:
        continue
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    tag = f"{organism.replace(' ', '_').replace('.', '')}_{short_name}"

    out_path = MULTI_CAT_OUT_DIR / f'test_pred__{tag}.csv'
    sub[['mol_idx','organism','metric_type','target_column',
         'y_true_log2','y_pred_log2','y_true_raw','y_pred_raw']].to_csv(out_path, index=False)

    yt, yp = sub['y_true_log2'].values, sub['y_pred_log2'].values
    cat_metrics_rows.append({
        'Approach': 'emb+categorical',
        'Organism': organism, 'Target': short_name,
        'N_test': len(sub),
        'MAE':  mean_absolute_error(yt, yp),
        'RMSE': np.sqrt(mean_squared_error(yt, yp)),
        'R2':   r2_score(yt, yp),
        'Pearson_r': pearsonr(yt, yp)[0],
    })
    print(f'  ✓ {out_path.name}  (N={len(sub)})')

test_df[['mol_idx','organism','metric_type','target_column',
         'y_true_log2','y_pred_log2','y_true_raw','y_pred_raw']].to_csv(
    MULTI_CAT_OUT_DIR / 'test_pred__ALL_targets_concat.csv', index=False
)

cat_metrics_df = pd.DataFrame(cat_metrics_rows)
cat_metrics_df.to_csv(MULTI_CAT_OUT_DIR / 'metrics_per_target.csv', index=False)

cat_overall = {
    'Approach': 'emb+categorical',
    'N_test_total': len(test_df),
    'MAE':  mae, 'RMSE': rmse, 'R2': r2, 'Pearson_r': pr,
}
pd.DataFrame([cat_overall]).to_csv(MULTI_CAT_OUT_DIR / 'metrics_overall.csv', index=False)
print(f'\n✓ {MULTI_CAT_OUT_DIR}')

molecular_embeddings dim = 256
Multi-task датасет: 432 пар (mol, target)
N_train=345  N_test=87

Overall: MAE=1.0065  RMSE=1.4829  R²=0.6362  Pearson=0.8082
  ✓ test_pred__S_aureus_ATCC_43300_MIC.csv  (N=30)
  ✓ test_pred__S_aureus_ATCC_43300_MBC.csv  (N=18)
  ✓ test_pred__E_coli_ATCC_25922_MIC.csv  (N=15)
  ✓ test_pred__E_coli_ATCC_25922_MBC.csv  (N=24)

✓ /Users/egorilin/Desktop/COMA/embeddings_cat_test_predictions


In [24]:
# %%
MULTI_OAI_OUT_DIR = BASE / 'embeddings_openai_test_predictions'
MULTI_OAI_OUT_DIR.mkdir(exist_ok=True)

# pairs_df, mol_emb_per_pair, y_full из предыдущей ячейки переиспользуем
label_emb_per_pair = np.stack(
    [class_embeddings[k] for k in pairs_df['class_key'].values]
).astype(np.float32)

X_full_oai = np.hstack([mol_emb_per_pair, label_emb_per_pair])
print(f'X_full_oai: {X_full_oai.shape}  ({emb_dim} mol + {openai_emb_dim} openai)')

# Тот же split (тот же seed, тот же размер → те же индексы)
train_idx_oai, test_idx_oai = train_test_split(
    np.arange(len(pairs_df)), test_size=TEST_SIZE, random_state=RANDOM_STATE
)
# sanity-check: train_idx_oai должен совпадать с train_idx из 5.6
assert np.array_equal(train_idx_oai, train_idx), 'split разъехался — проверь seed'

X_train_oai = X_full_oai[train_idx_oai]
X_test_oai  = X_full_oai[test_idx_oai]
y_train_oai = y_full[train_idx_oai]
y_test_oai  = y_full[test_idx_oai]
meta_test_oai = pairs_df.iloc[test_idx_oai].reset_index(drop=True)

print(f'N_train={len(train_idx_oai)}  N_test={len(test_idx_oai)}')

model_oai = CatBoostRegressor(**CB_KW)
model_oai.fit(X_train_oai, y_train_oai)
y_pred_oai = model_oai.predict(X_test_oai)

mae  = mean_absolute_error(y_test_oai, y_pred_oai)
rmse = np.sqrt(mean_squared_error(y_test_oai, y_pred_oai))
r2   = r2_score(y_test_oai, y_pred_oai)
pr, _ = pearsonr(y_test_oai, y_pred_oai)
print(f'\nOverall: MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  Pearson={pr:.4f}')

# --- Сохранение per-target и concat ---
test_df = meta_test_oai.copy()
test_df['y_true_log2'] = y_test_oai
test_df['y_pred_log2'] = y_pred_oai
test_df['y_true_raw']  = 2.0 ** y_test_oai
test_df['y_pred_raw']  = 2.0 ** y_pred_oai

oai_metrics_rows = []
for target_col in TARGET_COLUMNS:
    sub = test_df[test_df['target_column'] == target_col]
    if len(sub) < 2:
        continue
    short_name = target_col.split(' Activity ')[1].replace(', мг/л', '')
    organism   = target_col.split(' Activity ')[0]
    tag = f"{organism.replace(' ', '_').replace('.', '')}_{short_name}"

    out_path = MULTI_OAI_OUT_DIR / f'test_pred__{tag}.csv'
    sub[['mol_idx','organism','metric_type','target_column',
         'y_true_log2','y_pred_log2','y_true_raw','y_pred_raw']].to_csv(out_path, index=False)

    yt, yp = sub['y_true_log2'].values, sub['y_pred_log2'].values
    oai_metrics_rows.append({
        'Approach': 'emb+openai',
        'Organism': organism, 'Target': short_name,
        'N_test': len(sub),
        'MAE':  mean_absolute_error(yt, yp),
        'RMSE': np.sqrt(mean_squared_error(yt, yp)),
        'R2':   r2_score(yt, yp),
        'Pearson_r': pearsonr(yt, yp)[0],
    })
    print(f'  ✓ {out_path.name}  (N={len(sub)})')

test_df[['mol_idx','organism','metric_type','target_column',
         'y_true_log2','y_pred_log2','y_true_raw','y_pred_raw']].to_csv(
    MULTI_OAI_OUT_DIR / 'test_pred__ALL_targets_concat.csv', index=False
)

oai_metrics_df = pd.DataFrame(oai_metrics_rows)
oai_metrics_df.to_csv(MULTI_OAI_OUT_DIR / 'metrics_per_target.csv', index=False)

oai_overall = {
    'Approach': 'emb+openai',
    'N_test_total': len(test_df),
    'MAE':  mae, 'RMSE': rmse, 'R2': r2, 'Pearson_r': pr,
}
pd.DataFrame([oai_overall]).to_csv(MULTI_OAI_OUT_DIR / 'metrics_overall.csv', index=False)
print(f'\n✓ {MULTI_OAI_OUT_DIR}')

X_full_oai: (432, 1792)  (256 mol + 1536 openai)
N_train=345  N_test=87

Overall: MAE=0.9873  RMSE=1.3855  R²=0.6824  Pearson=0.8430
  ✓ test_pred__S_aureus_ATCC_43300_MIC.csv  (N=30)
  ✓ test_pred__S_aureus_ATCC_43300_MBC.csv  (N=18)
  ✓ test_pred__E_coli_ATCC_25922_MIC.csv  (N=15)
  ✓ test_pred__E_coli_ATCC_25922_MBC.csv  (N=24)

✓ /Users/egorilin/Desktop/COMA/embeddings_openai_test_predictions


In [28]:
# %%
# Все метрики per-target в один DataFrame
all_per_target = pd.concat([
    desc_metrics_df, emb_metrics_df,
    desc_cat_metrics_df, desc_oai_metrics_df,
    emb_cat_metrics_df,  emb_oai_metrics_df,
], ignore_index=True)

# Короткие коды классов: SA_MIC, SA_MBC, EC_MIC, EC_MBC
def short_class(row):
    org = 'SA' if row['Organism'].startswith('S.') else 'EC'
    return f"{org}_{row['Target']}"

all_per_target['Class'] = all_per_target.apply(short_class, axis=1)
CLASS_ORDER = ['SA_MIC', 'SA_MBC', 'EC_MIC', 'EC_MBC']
APPROACH_ORDER = ['descriptors', 'embeddings',
                  'desc+categorical', 'desc+openai',
                  'emb+categorical', 'emb+openai']

# --- Сводные таблицы по каждой метрике: строки=подходы, столбцы=4 класса ---
def pivot_metric(metric):
    pv = all_per_target.pivot_table(
        index='Approach', columns='Class', values=metric, aggfunc='first'
    )
    pv = pv.reindex(index=APPROACH_ORDER, columns=CLASS_ORDER)
    return pv

print('\n' + '='*70)
print('РАСКЛАД ПО КЛАССАМ:  SA_MIC | SA_MBC | EC_MIC | EC_MBC')
print('='*70)

for metric in ['MAE', 'RMSE', 'R2', 'Pearson_r']:
    pv = pivot_metric(metric)
    print(f'\n--- {metric} ---')
    print(pv.round(4).to_string())
    pv.to_csv(BASE / f'comparison_by_class__{metric}.csv')

# --- Общий overall (concat across all 4 targets) ---
overall = pd.DataFrame([
    desc_overall, emb_overall,
    desc_cat_overall, desc_oai_overall,
    emb_cat_overall,  emb_oai_overall,
])

print('\n' + '='*70)
print('OVERALL (concat across all 4 targets)')
print('='*70)
print(overall[['Approach','N_test_total','MAE','RMSE','R2','Pearson_r']].round(4).to_string(index=False))

# --- Сохранение ---
all_per_target.to_csv(BASE / 'comparison_per_target_LONG.csv', index=False)
overall.to_csv(BASE / 'comparison_overall.csv', index=False)
print(f'\n✓ {BASE}/comparison_per_target_LONG.csv')
print(f'✓ {BASE}/comparison_overall.csv')
print(f'✓ {BASE}/comparison_by_class__<MAE|RMSE|R2|Pearson_r>.csv')


РАСКЛАД ПО КЛАССАМ:  SA_MIC | SA_MBC | EC_MIC | EC_MBC

--- MAE ---
Class             SA_MIC  SA_MBC  EC_MIC  EC_MBC
Approach                                        
descriptors       1.4996  1.4771  1.6652  1.6357
embeddings        1.0034  1.3542  1.0102  0.9995
desc+categorical  1.3440  1.0184  0.4280  0.8942
desc+openai       1.2955  0.9903  0.3422  0.9870
emb+categorical   1.2209  1.0712  0.5699  0.9628
emb+openai        1.1432  1.0409  0.6004  0.9939

--- RMSE ---
Class             SA_MIC  SA_MBC  EC_MIC  EC_MBC
Approach                                        
descriptors       1.9907  1.9861  2.1581  2.2201
embeddings        1.3343  1.8623  1.3993  1.2601
desc+categorical  1.7807  1.4364  0.6601  1.3148
desc+openai       1.7591  1.4964  0.6188  1.4670
emb+categorical   1.9016  1.4569  0.8838  1.1709
emb+openai        1.6429  1.4227  0.8559  1.2683

--- R2 ---
Class             SA_MIC  SA_MBC  EC_MIC  EC_MBC
Approach                                        
descriptors      -0.237

In [22]:
# %%
summary = pd.concat([
    desc_metrics_df, emb_metrics_df, cat_metrics_df, oai_metrics_df
], ignore_index=True)
print('Per-target:')
print(summary.to_string(index=False))

overall = pd.DataFrame([desc_overall, emb_overall, cat_overall, oai_overall])
print('\nOverall (concat across all 4 targets):')
print(overall.to_string(index=False))

summary.to_csv(BASE / 'comparison_per_target.csv', index=False)
overall.to_csv(BASE / 'comparison_overall.csv', index=False)
print(f'\n✓ {BASE}/comparison_per_target.csv')
print(f'✓ {BASE}/comparison_overall.csv')

Per-target:
       Approach             Organism Target  N_train  N_test      MAE     RMSE        R2  Pearson_r
    descriptors S. aureus ATCC 43300    MIC     80.0      21 1.499614 1.990659 -0.237649   0.222565
    descriptors S. aureus ATCC 43300    MBC     80.0      21 1.477137 1.986071 -0.554022   0.047284
    descriptors   E. coli ATCC 25922    MIC     80.0      21 1.665188 2.158051 -0.043290   0.248387
    descriptors   E. coli ATCC 25922    MBC     80.0      21 1.635707 2.220074 -0.054160   0.242405
     embeddings S. aureus ATCC 43300    MIC     86.0      22 1.003393 1.334261  0.029682   0.456203
     embeddings S. aureus ATCC 43300    MBC     86.0      22 1.354210 1.862322  0.010676   0.445089
     embeddings   E. coli ATCC 25922    MIC     86.0      22 1.010180 1.399272  0.166473   0.611821
     embeddings   E. coli ATCC 25922    MBC     86.0      22 0.999486 1.260058  0.340718   0.672748
emb+categorical S. aureus ATCC 43300    MIC      NaN      22 0.940999 1.262394  0.131395

# Startify 

In [29]:
# %%
import time
from openai import OpenAI
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier, Pool

OPENROUTER_API_KEY = "[REDACTED_API_KEY]"
EMBEDDING_MODEL = "openai/text-embedding-3-small"

_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

def get_embedding(text, model=EMBEDDING_MODEL):
    text = str(text).replace("\n", " ")[:8000]
    r = _client.embeddings.create(input=[text], model=model)
    return np.array(r.data[0].embedding)

def get_embeddings_batch(texts, model=EMBEDDING_MODEL, batch_size=50):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t).replace("\n", " ")[:8000] for t in texts[i:i+batch_size]]
        try:
            r = _client.embeddings.create(input=batch, model=model)
            all_embs.extend([np.array(d.embedding) for d in r.data])
        except Exception as e:
            print(f'  ⚠ batch err @ {i}: {e} — fallback single')
            for t in batch:
                try:    all_embs.append(get_embedding(t, model))
                except: all_embs.append(np.zeros(1536))
        if (i + batch_size) % 200 == 0 or i + batch_size >= len(texts):
            print(f'    {min(i+batch_size, len(texts))}/{len(texts)} done')
    return np.array(all_embs)

# тест
_t = get_embedding("test")
LLM_DIM = len(_t)
print(f'✓ OpenRouter работает, LLM_DIM={LLM_DIM}')

# --- SMILES для каждой молекулы ---
SHORT_NAMES = ['SA_MIC', 'SA_MBC', 'EC_MIC', 'EC_MBC']

smiles_col = None
for c in target.columns:
    if 'smiles' in c.lower():
        smiles_col = c; break
if smiles_col is None and 'General formula' in target.columns:
    smiles_col = 'General formula'

if smiles_col:
    smiles_list = target[smiles_col].astype(str).tolist()
    print(f'✓ SMILES колонка: "{smiles_col}" ({len(smiles_list)} строк)')
else:
    print('⚠ SMILES колонки не нашёл, использую текстовое представление фичей')
    smiles_list = [
        f"Molecule {i}: features={list(data_drop.iloc[i, :10].round(3).values)}"
        for i in range(len(data_drop))
    ]

print('\nПолучение LLM эмбеддингов SMILES...')
t0 = time.time()
llm_mol_emb = get_embeddings_batch(smiles_list)
print(f'✓ llm_mol_emb: {llm_mol_emb.shape}  ({time.time()-t0:.1f}s)')

# --- LLM эмбеддинги описаний классов ---
class_descriptions_idx = {
    0: ("Minimum Inhibitory Concentration (MIC) against Staphylococcus aureus ATCC 43300, "
        "a methicillin-resistant Gram-positive bacterium. MIC is the lowest concentration "
        "that inhibits visible bacterial growth."),
    1: ("Minimum Bactericidal Concentration (MBC) against Staphylococcus aureus ATCC 43300, "
        "a methicillin-resistant Gram-positive bacterium. MBC is the lowest concentration "
        "that kills 99.9 percent of bacteria."),
    2: ("Minimum Inhibitory Concentration (MIC) against Escherichia coli ATCC 25922, "
        "a Gram-negative reference strain. MIC is the lowest concentration "
        "that inhibits visible bacterial growth."),
    3: ("Minimum Bactericidal Concentration (MBC) against Escherichia coli ATCC 25922, "
        "a Gram-negative reference strain. MBC is the lowest concentration "
        "that kills 99.9 percent of bacteria."),
}
class_emb_idx = {}
for tid, desc in class_descriptions_idx.items():
    class_emb_idx[tid] = get_embedding(desc)
    print(f'  {SHORT_NAMES[tid]}: dim={len(class_emb_idx[tid])}')
print('✓ LLM эмбеддинги классов готовы')

✓ OpenRouter работает, LLM_DIM=1536
✓ SMILES колонка: "General formula" (108 строк)

Получение LLM эмбеддингов SMILES...
    108/108 done
✓ llm_mol_emb: (108, 1536)  (6.7s)
  SA_MIC: dim=1536
  SA_MBC: dim=1536
  EC_MIC: dim=1536
  EC_MBC: dim=1536
✓ LLM эмбеддинги классов готовы


In [30]:
# %%
CONTR_DIM = molecular_embeddings.shape[1]
print(f'CONTR_DIM = {CONTR_DIM}, LLM_DIM = {LLM_DIM}')

# ---------- ЭМБЕДДИНГИ multi-task ----------
emb_rows = []
for task_idx, col in enumerate(TARGET_COLUMNS):
    sn = SHORT_NAMES[task_idx]
    valid = np.where(~target[col].isna().values)[0]
    for gi in valid:
        emb_rows.append({
            'task_idx':      task_idx,
            'short_name':    sn,
            'target_column': col,
            'global_i':      int(gi),
            'y_raw':         float(target[col].values[gi]),
            'y_log2':        float(np.log2(target[col].values[gi])),
        })
emb_md = pd.DataFrame(emb_rows).reset_index(drop=True)
print(f'Emb multi-task: {len(emb_md)} пар')
for sn in SHORT_NAMES:
    print(f'  {sn}: {(emb_md["short_name"]==sn).sum()}')

emb_contr  = molecular_embeddings[emb_md['global_i'].values]
emb_llmmol = llm_mol_emb[emb_md['global_i'].values]
emb_llmcls = np.stack([class_emb_idx[t] for t in emb_md['task_idx'].values])
emb_taskid = emb_md['task_idx'].values
emb_y_log2 = emb_md['y_log2'].values
emb_y_raw  = emb_md['y_raw'].values

# composite labels для классификации
emb_comp_le = LabelEncoder()
emb_y_cls = emb_comp_le.fit_transform(
    [f"{t}_{v:.1f}" for t, v in zip(emb_md['task_idx'].values, emb_y_raw)]
)
print(f'  composite classes (emb): {len(emb_comp_le.classes_)}')

# ---------- ДЕСКРИПТОРЫ multi-task ----------
desc_rows_list = []
for task_idx, col in enumerate(TARGET_COLUMNS):
    sn = SHORT_NAMES[task_idx]
    cols_needed = FEATURE_COLS + [col]
    m = desc_df[cols_needed].notna().all(axis=1)
    sub = desc_df.loc[m, cols_needed].copy()
    sub['task_idx']      = task_idx
    sub['short_name']    = sn
    sub['target_column'] = col
    sub['global_i']      = sub.index.values
    sub['y_raw']         = sub[col].values.astype(float)
    sub['y_log2']        = np.log2(sub[col].values)
    desc_rows_list.append(sub)
desc_md = pd.concat(desc_rows_list, ignore_index=True)
print(f'\nDesc multi-task: {len(desc_md)} пар')
for sn in SHORT_NAMES:
    print(f'  {sn}: {(desc_md["short_name"]==sn).sum()}')

desc_feats  = desc_md[FEATURE_COLS].values.astype(float)
desc_llmmol = llm_mol_emb[desc_md['global_i'].values]
desc_llmcls = np.stack([class_emb_idx[t] for t in desc_md['task_idx'].values])
desc_taskid = desc_md['task_idx'].values
desc_y_log2 = desc_md['y_log2'].values
desc_y_raw  = desc_md['y_raw'].values

desc_comp_le = LabelEncoder()
desc_y_cls = desc_comp_le.fit_transform(
    [f"{t}_{v:.1f}" for t, v in zip(desc_md['task_idx'].values, desc_y_raw)]
)
print(f'  composite classes (desc): {len(desc_comp_le.classes_)}')

CONTR_DIM = 256, LLM_DIM = 1536
Emb multi-task: 432 пар
  SA_MIC: 108
  SA_MBC: 108
  EC_MIC: 108
  EC_MBC: 108
  composite classes (emb): 42

Desc multi-task: 404 пар
  SA_MIC: 101
  SA_MBC: 101
  EC_MIC: 101
  EC_MBC: 101
  composite classes (desc): 42


In [31]:
# %%
def mt_regression(X, y, task_ids, name, cat_features=None):
    """Multi-task регрессия. Возвращает (overall_dict, per_class_df, yte, yp, tte, model)."""
    Xtr, Xte, ytr, yte, ttr, tte = train_test_split(
        X, y, task_ids, test_size=TEST_SIZE, random_state=RANDOM_STATE,
        stratify=task_ids,
    )
    if cat_features and isinstance(Xtr, pd.DataFrame):
        cb = CatBoostRegressor(**CB_KW)
        cb.fit(Pool(Xtr, label=ytr, cat_features=cat_features))
        yp = cb.predict(Pool(Xte, cat_features=cat_features))
    else:
        cb = CatBoostRegressor(**CB_KW)
        cb.fit(Xtr, ytr)
        yp = cb.predict(Xte)

    overall = {
        'Approach': name,
        'N_test':   len(yte),
        'MAE':      mean_absolute_error(yte, yp),
        'RMSE':     np.sqrt(mean_squared_error(yte, yp)),
        'R2':       r2_score(yte, yp),
        'Pearson_r': pearsonr(yte, yp)[0],
    }
    per_class = []
    for tid, sn in enumerate(SHORT_NAMES):
        m = tte == tid
        if m.sum() < 2: continue
        per_class.append({
            'Approach': name, 'Class': sn, 'N_test': int(m.sum()),
            'MAE':  mean_absolute_error(yte[m], yp[m]),
            'RMSE': np.sqrt(mean_squared_error(yte[m], yp[m])),
            'R2':   r2_score(yte[m], yp[m]),
            'Pearson_r': pearsonr(yte[m], yp[m])[0],
        })
    print(f'\n  {name}:')
    print(f'    OVERALL: R²={overall["R2"]:.4f}  MAE={overall["MAE"]:.4f}  Pearson={overall["Pearson_r"]:.4f}')
    for pc in per_class:
        print(f'    {pc["Class"]:8s}: R²={pc["R2"]:.4f}  MAE={pc["MAE"]:.4f}')
    return overall, pd.DataFrame(per_class), yte, yp, tte, cb


def mt_classification(X, y_cls, task_ids, name, cat_features=None):
    """Multi-task классификация по composite label."""
    Xtr, Xte, ytr, yte, ttr, tte = train_test_split(
        X, y_cls, task_ids, test_size=TEST_SIZE, random_state=RANDOM_STATE,
        stratify=task_ids,
    )
    if cat_features and isinstance(Xtr, pd.DataFrame):
        cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                                verbose=False, random_seed=RANDOM_STATE,
                                auto_class_weights='Balanced')
        cb.fit(Pool(Xtr, label=ytr, cat_features=cat_features))
        yp = cb.predict(Pool(Xte, cat_features=cat_features)).flatten().astype(int)
    else:
        cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                                verbose=False, random_seed=RANDOM_STATE,
                                auto_class_weights='Balanced')
        cb.fit(Xtr, ytr)
        yp = cb.predict(Xte).flatten().astype(int)

    overall = {
        'Approach': name,
        'N_test':   len(yte),
        'Accuracy': accuracy_score(yte, yp),
        'F1_weighted': f1_score(yte, yp, average='weighted', zero_division=0),
    }
    per_class = []
    for tid, sn in enumerate(SHORT_NAMES):
        m = tte == tid
        if m.sum() < 2: continue
        per_class.append({
            'Approach': name, 'Class': sn, 'N_test': int(m.sum()),
            'Accuracy':    accuracy_score(yte[m], yp[m]),
            'F1_weighted': f1_score(yte[m], yp[m], average='weighted', zero_division=0),
        })
    print(f'\n  {name}:')
    print(f'    OVERALL: Acc={overall["Accuracy"]:.4f}  F1w={overall["F1_weighted"]:.4f}')
    for pc in per_class:
        print(f'    {pc["Class"]:8s}: Acc={pc["Accuracy"]:.4f}  F1w={pc["F1_weighted"]:.4f}')
    return overall, pd.DataFrame(per_class)

In [ ]:
# %%
print('='*80); print('EMB MULTI-TASK'); print('='*80)

emb_overalls_reg = []
emb_per_class_reg = []
emb_overalls_cls = []
emb_per_class_cls = []

# 2) Contr + task_id (categorical)
Xdf = pd.DataFrame(emb_contr, columns=[f'ce_{i}' for i in range(CONTR_DIM)])
Xdf['task_id'] = emb_taskid.astype(str)
o, pc, *_ = mt_regression(Xdf, emb_y_log2, emb_taskid, 'emb: Contr + task_id', cat_features=['task_id'])
emb_overalls_reg.append(o); emb_per_class_reg.append(pc)
o, pc = mt_classification(Xdf, emb_y_cls, emb_taskid, 'emb: Contr + task_id', cat_features=['task_id'])
emb_overalls_cls.append(o); emb_per_class_cls.append(pc)

# 3) Contr + LLM_cls
X = np.hstack([emb_contr, emb_llmcls])
o, pc, *_ = mt_regression(X, emb_y_log2, emb_taskid, f'emb: Contr({CONTR_DIM}) + LLM_cls({LLM_DIM})')
emb_overalls_reg.append(o); emb_per_class_reg.append(pc)
o, pc = mt_classification(X, emb_y_cls, emb_taskid, f'emb: Contr({CONTR_DIM}) + LLM_cls({LLM_DIM})')
emb_overalls_cls.append(o); emb_per_class_cls.append(pc)

# 4) LLM_mol + LLM_cls
X = np.hstack([emb_llmmol, emb_llmcls])
o, pc, *_ = mt_regression(X, emb_y_log2, emb_taskid, f'emb: LLM_mol({LLM_DIM}) + LLM_cls({LLM_DIM})')
emb_overalls_reg.append(o); emb_per_class_reg.append(pc)
o, pc = mt_classification(X, emb_y_cls, emb_taskid, f'emb: LLM_mol({LLM_DIM}) + LLM_cls({LLM_DIM})')
emb_overalls_cls.append(o); emb_per_class_cls.append(pc)

# 5) Hybrid: Contr + LLM_mol + LLM_cls
X = np.hstack([emb_contr, emb_llmmol, emb_llmcls])
o, pc, yte_h, yp_h, tte_h, cb_emb_hybrid = mt_regression(
    X, emb_y_log2, emb_taskid, f'emb: Contr+LLM_mol+LLM_cls'
)
emb_overalls_reg.append(o); emb_per_class_reg.append(pc)
o, pc = mt_classification(X, emb_y_cls, emb_taskid, f'emb: Contr+LLM_mol+LLM_cls')
emb_overalls_cls.append(o); emb_per_class_cls.append(pc)

emb_reg_overall_df = pd.DataFrame(emb_overalls_reg)
emb_reg_pc_df      = pd.concat(emb_per_class_reg, ignore_index=True)
emb_cls_overall_df = pd.DataFrame(emb_overalls_cls)
emb_cls_pc_df      = pd.concat(emb_per_class_cls, ignore_index=True)

EMB MULTI-TASK

  emb: Contr + task_id:
    OVERALL: R²=0.7189  MAE=0.8354  Pearson=0.8565
    SA_MIC  : R²=0.7102  MAE=0.8615
    SA_MBC  : R²=0.6403  MAE=0.8079
    EC_MIC  : R²=0.4269  MAE=0.8249
    EC_MBC  : R²=0.7394  MAE=0.8462

  emb: Contr + task_id:
    OVERALL: Acc=0.1954  F1w=0.1911
    SA_MIC  : Acc=0.2273  F1w=0.1788
    SA_MBC  : Acc=0.1429  F1w=0.1595
    EC_MIC  : Acc=0.1818  F1w=0.1821
    EC_MBC  : Acc=0.2273  F1w=0.2424

  emb: Contr(256) + LLM_cls(1536):
    OVERALL: R²=0.7671  MAE=0.7337  Pearson=0.8798
    SA_MIC  : R²=0.7494  MAE=0.7414
    SA_MBC  : R²=0.6752  MAE=0.7211
    EC_MIC  : R²=0.5918  MAE=0.6965
    EC_MBC  : R²=0.7708  MAE=0.7754

  emb: Contr(256) + LLM_cls(1536):
    OVERALL: Acc=0.2184  F1w=0.2245
    SA_MIC  : Acc=0.1818  F1w=0.1303
    SA_MBC  : Acc=0.2381  F1w=0.2691
    EC_MIC  : Acc=0.2273  F1w=0.2525
    EC_MBC  : Acc=0.2273  F1w=0.2479

  emb: LLM_mol(1536) + LLM_cls(1536):
    OVERALL: R²=0.7313  MAE=0.7909  Pearson=0.8599
    SA_MIC  : R

In [ ]:
# %%
EXT_OUT_DIR = BASE / 'extended_comparison'
EXT_OUT_DIR.mkdir(exist_ok=True)

# --- Regression ---
reg_overall = pd.concat([emb_reg_overall_df, desc_reg_overall_df], ignore_index=True)
reg_pc      = pd.concat([emb_reg_pc_df, desc_reg_pc_df], ignore_index=True)

reg_pivot_r2  = reg_pc.pivot_table(index='Approach', columns='Class', values='R2',  aggfunc='first')
reg_pivot_mae = reg_pc.pivot_table(index='Approach', columns='Class', values='MAE', aggfunc='first')
reg_pivot_r2  = reg_pivot_r2.reindex(columns=SHORT_NAMES)
reg_pivot_mae = reg_pivot_mae.reindex(columns=SHORT_NAMES)

print('\n' + '='*70)
print('REGRESSION — R² по классам')
print('='*70)
print(reg_pivot_r2.round(4).to_string())

print('\nREGRESSION — MAE по классам')
print(reg_pivot_mae.round(4).to_string())

print('\nREGRESSION — Overall')
print(reg_overall[['Approach','N_test','MAE','RMSE','R2','Pearson_r']].round(4).to_string(index=False))

# --- Classification ---
cls_overall = pd.concat([emb_cls_overall_df, desc_cls_overall_df], ignore_index=True)
cls_pc      = pd.concat([emb_cls_pc_df, desc_cls_pc_df], ignore_index=True)

cls_pivot_acc = cls_pc.pivot_table(index='Approach', columns='Class', values='Accuracy',    aggfunc='first')
cls_pivot_f1  = cls_pc.pivot_table(index='Approach', columns='Class', values='F1_weighted', aggfunc='first')
cls_pivot_acc = cls_pivot_acc.reindex(columns=SHORT_NAMES)
cls_pivot_f1  = cls_pivot_f1.reindex(columns=SHORT_NAMES)

print('\n' + '='*70)
print('CLASSIFICATION — Accuracy по классам')
print('='*70)
print(cls_pivot_acc.round(4).to_string())

print('\nCLASSIFICATION — F1_weighted по классам')
print(cls_pivot_f1.round(4).to_string())

print('\nCLASSIFICATION — Overall')
print(cls_overall[['Approach','N_test','Accuracy','F1_weighted']].round(4).to_string(index=False))

# --- Сохранение ---
reg_overall.to_csv (EXT_OUT_DIR / 'regression_overall.csv', index=False)
reg_pc.to_csv      (EXT_OUT_DIR / 'regression_per_class.csv', index=False)
reg_pivot_r2.to_csv(EXT_OUT_DIR / 'regression_pivot_R2.csv')
reg_pivot_mae.to_csv(EXT_OUT_DIR / 'regression_pivot_MAE.csv')

cls_overall.to_csv (EXT_OUT_DIR / 'classification_overall.csv', index=False)
cls_pc.to_csv      (EXT_OUT_DIR / 'classification_per_class.csv', index=False)
cls_pivot_acc.to_csv(EXT_OUT_DIR / 'classification_pivot_Accuracy.csv')
cls_pivot_f1.to_csv (EXT_OUT_DIR / 'classification_pivot_F1w.csv')

# Excel summary
with pd.ExcelWriter(EXT_OUT_DIR / 'extended_comparison_SUMMARY.xlsx') as w:
    reg_overall.to_excel  (w, sheet_name='reg_overall',  index=False)
    reg_pc.to_excel       (w, sheet_name='reg_per_class', index=False)
    reg_pivot_r2.to_excel (w, sheet_name='reg_R2_pivot')
    reg_pivot_mae.to_excel(w, sheet_name='reg_MAE_pivot')
    cls_overall.to_excel  (w, sheet_name='cls_overall',  index=False)
    cls_pc.to_excel       (w, sheet_name='cls_per_class', index=False)
    cls_pivot_acc.to_excel(w, sheet_name='cls_Acc_pivot')
    cls_pivot_f1.to_excel (w, sheet_name='cls_F1w_pivot')

print(f'\n✓ Все таблицы сохранены в: {EXT_OUT_DIR}')
print(f'✓ Сводный Excel: {EXT_OUT_DIR / "extended_comparison_SUMMARY.xlsx"}')

# --- Feature importance (hybrid emb) ---
fi = cb_emb_hybrid.get_feature_importance()
contr_fi  = fi[:CONTR_DIM]
llmmol_fi = fi[CONTR_DIM:CONTR_DIM+LLM_DIM]
llmcls_fi = fi[CONTR_DIM+LLM_DIM:]
print('\n=== Feature Importance (emb hybrid) ===')
print(f'  Contrastive ({CONTR_DIM}d):  sum={contr_fi.sum():.2f}%  mean={contr_fi.mean():.4f}')
print(f'  LLM mol     ({LLM_DIM}d):  sum={llmmol_fi.sum():.2f}%  mean={llmmol_fi.mean():.4f}')
print(f'  LLM cls     ({LLM_DIM}d):  sum={llmcls_fi.sum():.2f}%  mean={llmcls_fi.mean():.4f}')

fi_df = pd.DataFrame({
    'block': ['Contrastive', 'LLM_mol', 'LLM_cls'],
    'dim':   [CONTR_DIM, LLM_DIM, LLM_DIM],
    'sum_importance_pct': [contr_fi.sum(), llmmol_fi.sum(), llmcls_fi.sum()],
    'mean_importance':    [contr_fi.mean(), llmmol_fi.mean(), llmcls_fi.mean()],
})
fi_df.to_csv(EXT_OUT_DIR / 'feature_importance_emb_hybrid.csv', index=False)
print(f'\n✓ feature_importance_emb_hybrid.csv')